# L11 · Integrated comparison and next research

## Goal

**Estimated time:** 45 min · **Path:** fast, full

- justify method selection
- compute the OPD2 delta
- separate avg@K from pass@K

### Current position: L10 → **L11** → finish

```text
Prompt/Data -> state source -> ... -> L11 -> ... -> fair evaluation
```

Alt text: The course map highlights L11 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L11"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L11', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

The final lesson chooses methods by assumptions instead of declaring one universal winner. Evaluate OPD2 delta and language retention separately from small-K efficiency and large-K capability boundaries.

Figure alt: labels and numbers remain readable without color.

### Core mechanics

OPD² uses the log-probability difference between a post-trained teacher and its base teacher, focusing on tokens made more preferred by post-training. Unlike single-teacher OPD, the `teacher - teacher_base` delta is central, so both models need aligned tokenizers and templates.

Test-time scaling separates `avg@K` (mean success among K samples) from `pass@K` (probability at least one succeeds). OPD may improve small-K efficiency while losing problems solvable at large K, so track gained/lost solvability and multilingual retention separately.

### Production implementation: why this design

OPD² centers the teacher/base delta and gates positive improvement regions. Test-time-scaling helpers accept a boolean `[problem, sample]` matrix, fixing metric definitions in code. These are a training-selection mechanism and an evaluation lens, not one loss.

Production code: [`opd2.py`](../../src/opd_study/algorithms/opd2.py), [`test_time_scaling.py`](../../src/opd_study/diagnostics/test_time_scaling.py).

In [2]:
import inspect
from opd_study.algorithms import opd2_loss
from opd_study.diagnostics import scaling_metrics

objects_to_show = (opd2_loss, scaling_metrics,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.algorithms.opd2.opd2_loss
def opd2_loss(
    student_logits: Tensor,
    trajectories: TrajectoryBatch,
    teacher_signals: TeacherSignals,
    teacher_base_signals: TeacherSignals,
    *,
    centering_top_k: int | None = None,
) -> LossOutput:
    """Centered teacher-minus-base delta advantage with OPD-direction gating.

    This follows the paper's three essential choices: delta reward, action-independent
    centering, and the joint-sign condition.  It is intentionally not presented as a
    reproduction of the official large-scale TRL/GRPO recipe.
    """

    teacher = teacher_signals.logits
    teacher_base = teacher_base_signals.logits
    if teacher is None or teacher_base is None:
        raise ValueError("OPD² requires teacher and teacher-base logits")
    if teacher.shape != student_logits.shape or teacher_base.shape != student_logits.shape:
        raise ValueError("teacher, teacher-base and student logits must match")
    shifted_student, target_ids, _, pred

### Alternatives and trade-offs

A practical rule: start with vanilla OPD for aligned single-turn support; consider vOPD for sampled variance, TCOD/SOD/SAGE for long horizons, and OPD² for post-training deltas. Decide against equal-budget SFT/KD baselines on held-out data.

### 2/3 · Run and observe

Predict before running: which invariant should you inspect first in L11's output? Write one sentence, then run.

In [3]:
from opd_study.algorithms import opd2_loss
from opd_study.data import CharacterTokenizer, collate_examples, generate_tiny_arithmetic
from opd_study.types import TeacherSignals

tokenizer = CharacterTokenizer(); rows = generate_tiny_arithmetic(train_rows=2, validation_rows=1, test_rows=1).train
batch = collate_examples(rows, tokenizer)
shape = (*batch.token_ids.shape, tokenizer.vocab_size)
student = torch.randn(shape, requires_grad=True)
teacher, teacher_base = torch.randn(shape), torch.randn(shape)
delta_output = opd2_loss(student, batch, TeacherSignals(logits=teacher),
    TeacherSignals(logits=teacher_base), centering_top_k=16)
print("OPD2 gate rate:", delta_output.metrics["opd2/gate_rate"])
print("Delta isolates teacher post-training change; multilingual retention must be evaluated separately.")

OPD2 gate rate: 0.529411792755127
Delta isolates teacher post-training change; multilingual retention must be evaluated separately.


In [4]:
from opd_study.diagnostics.test_time_scaling import gained_and_lost_solvability, scaling_metrics

before = torch.tensor([[1,0,0,0], [0,0,1,0], [0,0,0,0]], dtype=torch.bool)
after = torch.tensor([[0,0,0,0], [1,1,0,0], [1,0,0,0]], dtype=torch.bool)
for k in (1, 2, 4):
    metric = scaling_metrics(after, k=k)
    print(k, "avg@K", metric.avg_at_k, "pass@K", metric.pass_at_k)
print("solvability:", gained_and_lost_solvability(before, after, k=4))

1 avg@K 0.6666666865348816 pass@K 0.6666666865348816
2 avg@K 0.5 pass@K 0.6666666865348816
4 avg@K 0.25 pass@K 0.6666666865348816
solvability: {'gained': 1, 'lost': 1, 'retained': 1, 'never_solved': 0}


## Checks

In [5]:
metric = scaling_metrics(after, k=4)
assert metric.avg_at_k != metric.pass_at_k
changes = gained_and_lost_solvability(before, after, k=4)
assert sum(changes.values()) == before.shape[0]
assert 0 <= delta_output.metrics["opd2/gate_rate"] <= 1
print("check passed: delta gating and capability-boundary accounting are explicit")

check passed: delta gating and capability-boundary accounting are explicit


**Exercise (10 min):** write a five-sentence method memo covering task horizon, support, logit access, hardware, and the SFT baseline.

<details><summary>Check</summary>Assumptions and measurement precede the algorithm name; include an avg@K/pass@K or language-retention guardrail.</details>

## My recurring mistakes

### M1 — Swapping avg@K and pass@K

- Wrong: use mean sample success and any-success probability interchangeably.
- Why: their meanings diverge as K grows.
- Fix: compute both from the same sampling matrix.
- Related check: `test_avg_and_pass_at_k_are_not_interchangeable`

### M2 — Choosing a method from one benchmark

- Wrong: use math accuracy alone and ignore retention/large-K ability.
- Why: post-training capability can move.
- Fix: combine task, retention, support, and scaling guardrails.
- Related check: `test_opd2_gate_closes_when_teacher_equals_base`

## 60-second summary

1. justify method selection
2. compute the OPD2 delta
3. separate avg@K from pass@K

## Next Steps

Before the next notebook, rerun the assertions and record one prediction you revised.

### Sources

- [`opd2`](https://arxiv.org/abs/2607.15161v1) · `2607.15161v1` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`opd2_multilingual`](https://arxiv.org/abs/2608.05802v1) · `2608.05802v1` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`opd_test_time_scaling`](https://arxiv.org/abs/2608.11829v1) · `2608.11829v1` · license `arXiv-non-exclusive-distribution-1.0` · [audited manifest](../../docs/sources.yml)